In [1]:
from PySide6.QtCore import Qt
print("PySide6 OK")

PySide6 OK


In [2]:
import sys
print(sys.executable)


c:\Users\Lenovo\.conda\envs\rvm_gui\python.exe


In [3]:
import time
import cv2
from ultralytics import YOLO
%pip install PySide6
from PySide6.QtCore import Qt, QThread, Signal
from PySide6.QtGui import QImage, QPixmap
from PySide6.QtWidgets import QApplication, QWidget, QLabel, QPushButton, QVBoxLayout, QHBoxLayout

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Ensure required packages are installed in this notebook environment

# Install PySide6 in the current kernel (fixes "Import 'PySide6.QtCore' could not be resolved")

import sys, time
import cv2
from ultralytics import YOLO

from PySide6.QtCore import Qt, QThread, Signal
from PySide6.QtGui import QImage, QPixmap
from PySide6.QtWidgets import QApplication, QWidget, QLabel, QPushButton, QVBoxLayout, QHBoxLayout


# =========================
# Config (من الكود القديم)
# =========================
MODEL_PATH = "rvm_best_yolov8s.pt"
CAM_INDEX = 0

FRAME_W, FRAME_H = 640, 480
TARGET_FPS = 30

IMGSZ = 640
CONF = 0.50
IOU  = 0.50
MAX_DET = 50

INFER_EVERY_N_FRAMES = 1   # 2 أو 3 لو عايز سلاسة أعلى
WINDOW_TITLE = "RVM YOLO GUI (Start / Stop) - FPS + Optimized"


def open_camera():
    # CAP_DSHOW على ويندوز ممكن يقلل latency
    cap = cv2.VideoCapture(CAM_INDEX, cv2.CAP_DSHOW)

    cap.set(cv2.CAP_PROP_FRAME_WIDTH, FRAME_W)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, FRAME_H)
    cap.set(cv2.CAP_PROP_FPS, TARGET_FPS)

    # تقليل الـ buffering (مش دايمًا بيمشي مع كل كاميرا/درايفر)
    cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)

    return cap


class VideoWorker(QThread):
    frame_ready = Signal(QImage)
    status_ready = Signal(str)

    def __init__(self, model):
        super().__init__()
        self.model = model
        self.running = False

    def run(self):
        self.running = True
        cap = open_camera()

        if not cap.isOpened():
            self.status_ready.emit("❌ Failed to open camera.")
            self.running = False
            return

        # FPS trackers (smoothed) — زي القديم
        prev_time = time.time()
        fps_smooth = 0.0
        alpha = 0.1

        frame_count = 0
        last_annotated = None

        self.status_ready.emit("✅ Detection running... (Press STOP to end)")

        while self.running:
            ret, frame = cap.read()
            if not ret:
                self.status_ready.emit("❌ Camera read failed.")
                break

            frame_count += 1
            do_infer = (frame_count % INFER_EVERY_N_FRAMES == 0)

            if do_infer:
                results = self.model.predict(
                    source=frame,
                    imgsz=IMGSZ,
                    conf=CONF,
                    iou=IOU,
                    max_det=MAX_DET,
                    verbose=False
                )
                annotated = results[0].plot()
                last_annotated = annotated
            else:
                annotated = last_annotated if last_annotated is not None else frame

            # FPS calc (smoothed)
            curr_time = time.time()
            dt = curr_time - prev_time
            prev_time = curr_time
            instant_fps = (1.0 / dt) if dt > 0 else 0.0
            fps_smooth = (1 - alpha) * fps_smooth + alpha * instant_fps

            # Overlay text (نفس سطر القديم)
            overlay = annotated.copy()
            cv2.putText(
                overlay,
                f"FPS: {fps_smooth:.1f} | imgsz={IMGSZ} | conf={CONF} | inferEvery={INFER_EVERY_N_FRAMES} | max_det={MAX_DET}",
                (15, 30),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.65,
                (0, 255, 0),
                2,
                cv2.LINE_AA
            )

            # Convert BGR -> RGB -> QImage
            rgb = cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)
            h, w, ch = rgb.shape
            qimg = QImage(rgb.data, w, h, ch * w, QImage.Format_RGB888)

            self.frame_ready.emit(qimg)

        cap.release()
        self.running = False
        self.status_ready.emit("🛑 Detection stopped.")

    def stop(self):
        self.running = False
        self.wait()


class RVM_GUI(QWidget):
    def __init__(self):
        super().__init__()
        self.setWindowTitle(WINDOW_TITLE)
        self.resize(950, 650)

        # Load model ONCE
        self.model = YOLO(MODEL_PATH)

        # UI
        self.video_label = QLabel("Press START to begin detection")
        self.video_label.setAlignment(Qt.AlignCenter)
        self.video_label.setStyleSheet(
            "background:#111; color:#ddd; border-radius:10px; padding:10px;"
        )

        self.status_label = QLabel("Status: Idle")
        self.status_label.setStyleSheet("font-size: 15px;")

        self.btn_start = QPushButton("Start Detection")
        self.btn_stop  = QPushButton("Stop Detection")

        self.btn_start.clicked.connect(self.start_detection)
        self.btn_stop.clicked.connect(self.stop_detection)

        btn_row = QHBoxLayout()
        btn_row.addWidget(self.btn_start)
        btn_row.addWidget(self.btn_stop)

        layout = QVBoxLayout()
        layout.addWidget(self.video_label)
        layout.addWidget(self.status_label)
        layout.addLayout(btn_row)
        self.setLayout(layout)

        self.worker = None

    def start_detection(self):
        if self.worker and self.worker.isRunning():
            return

        self.worker = VideoWorker(self.model)
        self.worker.frame_ready.connect(self.update_frame)
        self.worker.status_ready.connect(self.update_status)
        self.worker.start()

    def stop_detection(self):
        if self.worker and self.worker.isRunning():
            self.worker.stop()
        self.update_status("Status: Idle")

    def update_frame(self, qimg: QImage):
        pix = QPixmap.fromImage(qimg)
        self.video_label.setPixmap(
            pix.scaled(
                self.video_label.width(),
                self.video_label.height(),
                Qt.KeepAspectRatio,
                Qt.SmoothTransformation
            )
        )

    def update_status(self, text: str):
        self.status_label.setText(text)

    def closeEvent(self, event):
        self.stop_detection()
        event.accept()


# ===== RUN GUI =====
# ملاحظة: لو شغّلت الخلية دي مرة تانية بعد ما تقفل الواجهة، اعمل Restart Kernel
app = QApplication.instance() or QApplication(sys.argv)
window = RVM_GUI()
window.show()


: 